# Supervised fine-tuning of mini-GPT on Kaggle

Required notebook settings: accelerator **GPU T4**, **Internet on**, and the dataset attached. It must contain `sft_train.npz`, `sft_val.npz`, `tokenizer.json` and `pretrain_best.pt` (the pre-training checkpoint).

In [ ]:
import os

REPO_URL = "https://github.com/bamont/minigpt.git"
REPO_DIR = "/kaggle/working/mini-gpt"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

In [ ]:
DATA_DIR = "/kaggle/input/datasets/bbbaaammm/minigpt-tinystories/minigpt-tinystories"
RUN_DIR = "/kaggle/working/runs/sft"

print(os.listdir(DATA_DIR))

In [ ]:
import torch

print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
!nvidia-smi

In [ ]:
!python -m minigpt.training.sft --resume --override data.train={DATA_DIR}/sft_train.npz data.val={DATA_DIR}/sft_val.npz data.tokenizer={DATA_DIR}/tokenizer.json init.checkpoint={DATA_DIR}/pretrain_best.pt train.out_dir={RUN_DIR}

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{RUN_DIR}/metrics.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]
train = [r for r in records if r["type"] == "train"]
evals = [r for r in records if r["type"] == "eval"]

plt.figure(figsize=(6, 4))
plt.plot([r["step"] for r in train], [r["loss"] for r in train], label="train")
plt.plot([r["step"] for r in evals], [r["val_loss"] for r in evals], label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/curves.png", dpi=150)
plt.show()